In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introduzione al servizio di transpiler AI-powered di Qiskit

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Utilizzo stimato di QPU: Nessuno (NOTA: Questo tutorial non esegue job perché è focalizzato sulla transpilazione)*

## Contesto
Il **servizio di transpiler AI-powered di Qiskit (QTS)** introduce ottimizzazioni basate su machine learning sia nei pass di routing che di sintesi. Queste modalità AI sono state progettate per affrontare le limitazioni della transpilazione tradizionale, in particolare per circuiti su larga scala e topologie hardware complesse.

A partire da **luglio 2025**, il **Transpiler Service** è stato migrato sulla nuova piattaforma IBM Quantum&reg; e non è più disponibile. Per gli ultimi aggiornamenti sullo stato del Transpiler Service, consultate la [documentazione del servizio di transpiler](/guides/qiskit-transpiler-service). Potete comunque utilizzare il transpiler AI localmente, in modo simile alla transpilazione standard di Qiskit. È sufficiente sostituire `generate_preset_pass_manager()` con `generate_ai_pass_manager()`. Questa funzione costruisce un pass manager che integra i pass di routing e sintesi AI-powered direttamente nel vostro flusso di lavoro di transpilazione locale.

### Caratteristiche principali dei pass AI
- Pass di routing: Il routing AI-powered può regolare dinamicamente i percorsi dei qubit in base al circuito specifico e al backend, riducendo la necessità di eccessivi gate SWAP.
    - `AIRouting`: Selezione del layout e routing del circuito

- Pass di sintesi: Le tecniche AI ottimizzano la decomposizione di gate multi-qubit, minimizzando il numero di gate a due qubit, che sono tipicamente più soggetti a errori.
    - `AICliffordSynthesis`: Sintesi di gate Clifford
    - `AILinearFunctionSynthesis`: Sintesi di circuiti di funzioni lineari
    - `AIPermutationSynthesis`: Sintesi di circuiti di permutazione
    - `AIPauliNetworkSynthesis`: Sintesi di circuiti di Pauli Network (disponibile solo nel Qiskit Transpiler Service, non nell'ambiente locale)

- Confronto con la transpilazione tradizionale: Il transpiler standard di Qiskit è uno strumento robusto in grado di gestire efficacemente un ampio spettro di circuiti quantistici. Tuttavia, quando i circuiti crescono in scala o le configurazioni hardware diventano più complesse, i pass AI possono fornire ulteriori guadagni di ottimizzazione. Utilizzando modelli appresi per il routing e la sintesi, QTS perfeziona ulteriormente i layout dei circuiti e riduce l'overhead per compiti quantistici complessi o su larga scala.

Questo tutorial valuta le modalità AI utilizzando sia i pass di routing che di sintesi, confrontando i risultati con la transpilazione tradizionale per evidenziare dove l'AI offre guadagni di prestazione.

Per maggiori dettagli sui pass AI disponibili, consultate la [documentazione dei pass AI](/guides/ai-transpiler-passes).

### Perché utilizzare l'AI per la transpilazione dei circuiti quantistici?
Man mano che i circuiti quantistici crescono in dimensione e complessità, i metodi di transpilazione tradizionali faticano a ottimizzare i layout e ridurre efficacemente il numero di gate. I circuiti più grandi, in particolare quelli che coinvolgono centinaia di qubit, pongono sfide significative per il routing e la sintesi a causa dei vincoli del dispositivo, della connettività limitata e dei tassi di errore dei qubit.

È qui che la transpilazione AI-powered offre una soluzione potenziale. Sfruttando tecniche di machine learning, il transpiler AI-powered in Qiskit può prendere decisioni più intelligenti riguardo al routing dei qubit e alla sintesi dei gate, portando a una migliore ottimizzazione dei circuiti quantistici su larga scala.

### Brevi risultati di benchmarking
![Graph showing AI transpiler performance against Qiskit](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

Nei test di benchmarking, il transpiler AI ha prodotto costantemente circuiti più superficiali e di qualità superiore rispetto al transpiler standard di Qiskit. Per questi test, abbiamo utilizzato la strategia predefinita del pass manager di Qiskit, configurata con [`generate_preset_passmanager`]. Sebbene questa strategia predefinita sia spesso efficace, può avere difficoltà con circuiti più grandi o più complessi. Al contrario, i pass AI-powered hanno ottenuto una riduzione media del 24% nel numero di gate a due qubit e una riduzione del 36% nella profondità del circuito per circuiti grandi (100+ qubit) durante la transpilazione alla topologia heavy-hex dell'hardware IBM Quantum. Per maggiori informazioni su questi benchmark, consultate questo [blog](https://www.ibm.com/quantum/blog/qiskit-performance).

Questo tutorial esplora i principali vantaggi dei pass AI e come si confrontano con i metodi tradizionali.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## Requisiti

Prima di iniziare questo tutorial, assicuratevi di avere installato quanto segue:

* Qiskit SDK v1.0 o successivo, con supporto per la [visualizzazione](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 o successivo
* Qiskit IBM&reg; Transpiler con modalità locale AI (`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## Configurazione

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts: {qubit_range_sim}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Parte I. Pattern di Qiskit
Vediamo ora come utilizzare il servizio di transpiler AI con un semplice circuito quantistico, utilizzando i pattern di Qiskit. La chiave consiste nel creare un `PassManager` con `generate_ai_pass_manager()` invece del `generate_preset_pass_manager()` standard.
## Passo 1: Mappare gli input classici in un problema quantistico
In questa sezione, testeremo il transpiler AI sul circuito `efficient_su2`, un ansatz hardware-efficient ampiamente utilizzato. Questo circuito è particolarmente rilevante per gli algoritmi quantistici variazionali (ad esempio, VQE) e i compiti di machine learning quantistico, rendendolo un caso di test ideale per valutare le prestazioni di transpilazione.

Il circuito `efficient_su2` consiste in strati alternati di rotazioni a singolo qubit e gate di entanglement come i CNOT. Questi strati consentono un'esplorazione flessibile dello spazio degli stati quantistici mantenendo la profondità del gate gestibile. Ottimizzando questo circuito, miriamo a ridurre il numero di gate, migliorare la fedeltà e minimizzare il rumore. Questo lo rende un candidato forte per testare l'efficienza del transpiler AI.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### Creare pass manager AI e tradizionali
Per valutare l'efficacia del transpiler AI, eseguiremo due esecuzioni di transpilazione. Prima, traspilaremo il circuito utilizzando il transpiler AI. Poi, eseguiremo un confronto transpilando lo stesso circuito senza il transpiler AI, utilizzando metodi tradizionali. Entrambi i processi di transpilazione utilizzeranno la stessa mappa di accoppiamento dal backend scelto e il livello di ottimizzazione impostato a 3 per un confronto equo.

Entrambi questi metodi riflettono l'approccio standard per creare istanze di `PassManager` per transpilare circuiti in Qiskit.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

In questo test, confrontiamo le prestazioni del transpiler AI e del metodo di transpilazione standard sul circuito efficient_su2. Il transpiler AI raggiunge una profondità del circuito notevolmente inferiore mantenendo un numero di gate simile.

- **Profondità del circuito:** Il transpiler AI produce un circuito con una profondità a due qubit inferiore. Questo è previsto, poiché i pass AI sono addestrati per ottimizzare la profondità apprendendo i pattern di interazione dei qubit e sfruttando la connettività hardware in modo più efficace rispetto alle euristiche basate su regole.

- **Numero di gate:** Il numero totale di gate rimane simile tra i due metodi. Ciò è in linea con le aspettative poiché la transpilazione standard basata su SABRE minimizza esplicitamente il numero di swap, che domina l'overhead dei gate. Il transpiler AI invece dà priorità alla profondità complessiva e può occasionalmente scambiare alcuni gate aggiuntivi per un percorso di esecuzione più breve.

- **Tempo di transpilazione:** Il transpiler AI richiede più tempo per l'esecuzione rispetto al metodo standard. Ciò è dovuto al costo computazionale aggiuntivo derivante dall'invocazione di modelli appresi durante il routing e la sintesi. Al contrario, il transpiler basato su SABRE è ora significativamente più veloce dopo essere stato riscritto e ottimizzato in Rust, fornendo un routing euristico altamente efficiente su larga scala.

È importante notare che questi risultati si basano su un solo circuito. Per ottenere una comprensione completa di come il transpiler AI si confronta con i metodi tradizionali, è necessario testare una varietà di circuiti. Le prestazioni di QTS possono variare notevolmente a seconda del tipo di circuito che viene ottimizzato. Per un confronto più ampio, consultate i benchmark sopra o visitate il [blog.](https://www.ibm.com/quantum/blog/qiskit-performance)
## Passo 3: Eseguire utilizzando le primitive di Qiskit
Poiché questo tutorial si concentra sulla transpilazione, nessun esperimento verrà eseguito sul dispositivo quantistico. L'obiettivo è sfruttare le ottimizzazioni del Passo 2 per ottenere un circuito transpilato con profondità o numero di gate ridotti.
## Passo 4: Post-elaborare e restituire il risultato nel formato classico desiderato
Poiché non c'è esecuzione per questo notebook, non ci sono risultati da post-elaborare.
# Parte II. Analizzare e fare benchmark dei circuiti transpilati
In questa sezione, dimostreremo come analizzare il circuito transpilato e fare il suo benchmark rispetto alla versione originale in modo più dettagliato. Ci concentreremo su metriche come la profondità del circuito, il numero di gate e il tempo di transpilazione per valutare l'efficacia dell'ottimizzazione. Inoltre, discuteremo di come i risultati possano differire tra vari tipi di circuiti, offrendo approfondimenti sulle prestazioni più ampie del transpiler in diversi scenari.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts: {qubit_range_hw}"
)

Created 25 circuits with qubit counts: [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>